In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# =========================
# 1. Load training datasets
# =========================
water_quality = pd.read_csv("../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../data/terraclimate_features_training.csv")

# =========================
# 2. Convert dates
# =========================
water_quality["Sample Date"] = pd.to_datetime(water_quality["Sample Date"], dayfirst=True)
landsat["Sample Date"] = pd.to_datetime(landsat["Sample Date"], dayfirst=True)
terraclimate["Sample Date"] = pd.to_datetime(terraclimate["Sample Date"], dayfirst=True)

# =========================
# 3. Create temporal features
# =========================
water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear

# =========================
# 4. Merge training datasets
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 5. Feature engineering
# =========================
df["nir_swir16_ratio"] = df["nir"] / df["swir16"]
df["nir_swir22_ratio"] = df["nir"] / df["swir22"]
df["green_nir_ratio"] = df["green"] / df["nir"]

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / df["swir22"]

# Nuevas features v3
df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]
df["ndmi_day"] = df["NDMI"] * df["dayofyear"]

# =========================
# 6. Handle missing values
# =========================
df.fillna(df.median(numeric_only=True), inplace=True)

# =========================
# 7. Define training features and targets
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

X = df.drop(columns=targets + ["Sample Date", "Latitude", "Longitude"])
y = df[targets]

# =========================
# 8. Train final model
# =========================
rf_final = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_final.fit(X, y)

# =========================
# 9. Load submission datasets
# =========================
submission = pd.read_csv("../data/submission_template.csv")
landsat_val = pd.read_csv("../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../data/terraclimate_features_validation.csv")

# =========================
# 10. Convert dates
# =========================
submission["Sample Date"] = pd.to_datetime(submission["Sample Date"], dayfirst=True)
landsat_val["Sample Date"] = pd.to_datetime(landsat_val["Sample Date"], dayfirst=True)
terraclimate_val["Sample Date"] = pd.to_datetime(terraclimate_val["Sample Date"], dayfirst=True)

# =========================
# 11. Create temporal features
# =========================
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear

# =========================
# 12. Merge validation datasets
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 13. Same feature engineering
# =========================
df_val["nir_swir16_ratio"] = df_val["nir"] / df_val["swir16"]
df_val["nir_swir22_ratio"] = df_val["nir"] / df_val["swir22"]
df_val["green_nir_ratio"] = df_val["green"] / df_val["nir"]

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / df_val["swir22"]

# Nuevas features v3
df_val["nir_pet"] = df_val["nir"] * df_val["pet"]
df_val["swir16_pet"] = df_val["swir16"] * df_val["pet"]
df_val["ndmi_day"] = df_val["NDMI"] * df_val["dayofyear"]

# =========================
# 14. Handle missing values
# =========================
df_val.fillna(df_val.median(numeric_only=True), inplace=True)

# =========================
# 15. Prepare validation features
# =========================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

# =========================
# 16. Predict
# =========================
predictions = rf_final.predict(X_val)

# =========================
# 17. Build submission
# =========================
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

submission_v3 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

# =========================
# 18. Export
# =========================
submission_v3.to_csv("../submissions/submission_v3.csv", index=False)

# =========================
# 19. Quick check
# =========================
print(submission_v3.shape)
print(submission_v3.head())

(200, 6)
   Longitude   Latitude Sample Date  Total Alkalinity  Electrical Conductance  \
0  27.822778 -32.043333  2014-09-01        125.392410               472.75265   
1  26.077500 -33.329167  2015-09-16        105.388115               374.22025   
2  27.640028 -32.991639  2015-05-07         62.613045               475.88925   
3  24.439167 -34.096389  2012-02-07         65.528815               224.17240   
4  28.581667 -32.000556  2014-10-01        109.854635               570.60610   

   Dissolved Reactive Phosphorus  
0                         34.685  
1                         38.905  
2                         34.525  
3                         16.925  
4                         27.475  
